[Reference](https://blog.stackademic.com/i-built-a-tiny-gpt-from-scratch-in-under-40-lines-of-python-heres-the-exact-tutorial-2a0f461233e5$0)

# Step 1: Set up your environment


In [1]:
pip install torch

# Step 2: Turn text into numbers (tokenization)


In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
text = "hello world this is a tiny example of how a language model learns to predict the next character in a sequence"
# find every unique character in our text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# build a lookup table in both directions
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]        # string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l])  # list of ints -> string

# Step 3: Split your data and build batches


In [3]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [4]:
block_size = 8   # how many characters of context the model sees at once
batch_size = 4   # how many independent examples we train on at once

In [5]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

# Step 4: Build the model itself


In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (batch, time, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]              # only care about the last position
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Step 5: Train it and watch the loss drop


In [8]:
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(500):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print("final loss:", loss.item())

final loss: 1.6234544515609741


# Step 6: Generate text from your trained model


In [9]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, max_new_tokens=50)[0].tolist()))

 wofy thact is amixtelaremhe thodis yplworexagtho o
